<a href="https://colab.research.google.com/github/parthag1201/RAG-ify/blob/main/rag_from_scratch_P1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part 1 : Implementing RAG Pipeline using langchain , langsmith for monitoring dashboard and Google Gemini API

In [1]:
# (1) Install required packages (if missing)
! pip install google-generativeai langchain_google_genai chromadb langchain
! pip install langchain_community tiktoken langchain-openai langchainhub

INFO: pip is looking at multiple versions of langchain-google-genai to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 3.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 94.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 76.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.9/94.9 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.2/284.2 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 70.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.6/101.6 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 100.6 MB/s eta 0:00

In [2]:
# (2) Import Gemini components
import google.generativeai as genai
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

In [3]:
from google.colab import userdata # For API Secret

In [4]:
# LangSmith Configuration
import os
from langsmith import traceable   ## To use @traceable on llm calls
os.environ['LANGCHAIN_TRACING_V2'] = 'true'
os.environ['LANGCHAIN_ENDPOINT'] = 'https://api.smith.langchain.com'
os.environ['LANGCHAIN_API_KEY'] = userdata.get('LANGCHAIN_API_KEY')
# LANGCHAIN_API_KEY = userdata.get('LANGCHAIN_API_KEY')
os.environ['LANGSMITH_PROJECT']='Rag-from-scratch_P1'

In [5]:
from langsmith import utils
utils.tracing_is_enabled()

True

In [6]:
# LangChain Libraries
import bs4
from langchain import hub
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [7]:
from google.colab import auth
auth.authenticate_user()

In [8]:
# (3) Configure API keys
import os
os.environ['GOOGLE_API_KEY']=userdata.get('Gemini_API')
# GOOGLE_API_KEY = userdata.get('Gemini_API')  # Replace with actual key

# (4) Initialize Gemini components
embeddings = GoogleGenerativeAIEmbeddings(model="models/embedding-001")
llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0)

# Load Documents
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)
docs = loader.load()

In [9]:
print(docs)

[Document(metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/'}, page_content='\n\n      LLM Powered Autonomous Agents\n    \nDate: June 23, 2023  |  Estimated Reading Time: 31 min  |  Author: Lilian Weng\n\n\nBuilding agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.\nAgent System Overview#\nIn a LLM-powered autonomous agent system, LLM functions as the agent’s brain, complemented by several key components:\n\nPlanning\n\nSubgoal and decomposition: The agent breaks down large tasks into smaller, manageable subgoals, enabling efficient handling of complex tasks.\nReflection and refinement: The agent can do self-criticism and self-reflection over past actions, learn from mistake

In [10]:
# Split
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(docs)

print(splits)

[Document(metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/'}, page_content='LLM Powered Autonomous Agents\n    \nDate: June 23, 2023  |  Estimated Reading Time: 31 min  |  Author: Lilian Weng\n\n\nBuilding agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.\nAgent System Overview#\nIn a LLM-powered autonomous agent system, LLM functions as the agent’s brain, complemented by several key components:\n\nPlanning\n\nSubgoal and decomposition: The agent breaks down large tasks into smaller, manageable subgoals, enabling efficient handling of complex tasks.\nReflection and refinement: The agent can do self-criticism and self-reflection over past actions, learn from mistakes and refi

In [11]:
# Rest of the code remains same until vectorstore initialization
vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings,  # Using Gemini embeddings
    # persist_directory="./chrome_db"
)

retriever = vectorstore.as_retriever()

In [12]:
from langchain.prompts import ChatPromptTemplate, PromptTemplate  # Import from correct submodule
from langchain import LLMChain

In [13]:
# (5) Update prompt template for Gemini compatibility
prompt_template = """Answer the question based only on the context:
Context: {context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(prompt_template)
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# (6) Create RAG chain with Gemini
# traceable
# def rag(context,question,prompt,llm):
#   rag_chain = (
#     {"context": retriever | format_docs, "question": RunnablePassthrough()}
#     | prompt
#     | llm
#     | StrOutputParser()
#   )

# # Query execution remains same
#   rag_chain.invoke("What is a Task?")

In [14]:
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
  )

rag_chain.invoke("What is a Task?")

'Based on the context provided, a Task is an action performed by an expert model, such as deleting a file, searching files, analyzing code, getting improved code, writing tests, executing a Python file, generating an image, sending a tweet, doing nothing, or completing a task. The execution of these tasks is logged.'

# Part 2 : Indexing

In [15]:
# Doc
question = "What kinds of pets do I like?"
document = "My favorite pet is a cat."

In [19]:
# ENCODING STRING : " To convert number into a vector array"

## Get the number of tokens of a doc
import tiktoken

def num_tokens_from_string(string, encoding_name) -> int:
    """Returns the number of tokens in a text string."""
    encoding = tiktoken.get_encoding(encoding_name)
    num_tokens = len(encoding.encode(string))
    # print(encoding.encode(string))
    return num_tokens

num_tokens_from_string(question, "cl100k_base")

[3923, 13124, 315, 26159, 656, 358, 1093, 30]


8

In [21]:
# EMBEDDING

## Why ? Because embedding is basically converting text to numerical vector representations to capture a closer semantic meaning of the text.
## LLMs can't understand long sentences easily that's why we will split them into smaller chunks and then put them in a vector space.
## We can retrive these vectors (mapped according to indexes) using semantic of the prompts later.

embd = GoogleGenerativeAIEmbeddings(model="models/embedding-001")
query_result = embd.embed_query(question)
document_result = embd.embed_query(document)
# print(query_result)
len(query_result)

[-0.006222310476005077, -0.006061272229999304, -0.026451561599969864, -0.020352467894554138, 0.0056349425576627254, -0.008037385530769825, 0.028191709890961647, -0.010441510006785393, 0.027259130030870438, 0.013420348055660725, 0.06445972621440887, -0.016194358468055725, 0.025595178827643394, 0.016282696276903152, 0.0048562223091721535, -0.031569819897413254, 0.005982452072203159, -0.00033394136698916554, 0.0012122254120185971, -0.04223364591598511, 0.009387334808707237, -0.002535228617489338, -0.019733130931854248, -0.005538017023354769, 0.04108477756381035, -0.06169067695736885, 0.04840322211384773, -0.029915019869804382, 0.0035526345018297434, 0.04842689633369446, -0.06559431552886963, 0.05941709503531456, -0.07555248588323593, -0.0007875484297983348, -0.023788658902049065, -0.04315219447016716, -0.032032158225774765, -0.019618835300207138, -0.014419357292354107, 0.05566913262009621, 0.01879522204399109, 0.002243547234684229, -0.05868314951658249, -0.051063422113657, 0.0222663395106

768

In [22]:
# COSINE SIMILARITY : "Used after embedding to find most relevant matches from the created vector spaces"

## Vectors pointing in same direction will be having same context - even if words aren't the same
## Cosine_Output = A (Dot) B / |A|*|B|

import numpy as np

def cosine_similarity(vec1, vec2):
    dot_product = np.dot(vec1, vec2)
    norm_vec1 = np.linalg.norm(vec1)
    norm_vec2 = np.linalg.norm(vec2)
    return dot_product / (norm_vec1 * norm_vec2)

similarity = cosine_similarity(query_result, document_result)
print("Cosine Similarity:", similarity)

Cosine Similarity: 0.8535652119095083


In [24]:
#### INDEXING ####

# Load blog using LangChain Document Loader
import bs4
from langchain_community.document_loaders import WebBaseLoader
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)
blog_docs = loader.load()

In [25]:
# SPLIT : "We try to split on them in order until the chunks are small enough. The default list is ["\n\n", "\n", " ", ""]"
from langchain.text_splitter import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=300,
    chunk_overlap=50)

# Make splits
splits = text_splitter.split_documents(blog_docs)
# print(splits)

[Document(metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/'}, page_content='LLM Powered Autonomous Agents\n    \nDate: June 23, 2023  |  Estimated Reading Time: 31 min  |  Author: Lilian Weng\n\n\nBuilding agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.\nAgent System Overview#\nIn a LLM-powered autonomous agent system, LLM functions as the agent’s brain, complemented by several key components:\n\nPlanning\n\nSubgoal and decomposition: The agent breaks down large tasks into smaller, manageable subgoals, enabling efficient handling of complex tasks.\nReflection and refinement: The agent can do self-criticism and self-reflection over past actions, learn from mistakes and refi

In [27]:
# Indexing : Into the vector store and retriever
from langchain_community.vectorstores import Chroma
vectorstore = Chroma.from_documents(documents=splits,
                                    embedding=GoogleGenerativeAIEmbeddings(model="models/embedding-001"))

retriever = vectorstore.as_retriever()

# Part 3 : Retrieval

In [28]:
# Index
from langchain_community.vectorstores import Chroma
vectorstore = Chroma.from_documents(documents=splits,
                                    embedding=GoogleGenerativeAIEmbeddings(model="models/embedding-001"))


retriever = vectorstore.as_retriever(search_kwargs={"k": 1})

In [42]:
docs = retriever.invoke("What is Task Decomposition?")

In [43]:
print(docs)

[Document(metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/'}, page_content='Fig. 1. Overview of a LLM-powered autonomous agent system.\nComponent One: Planning#\nA complicated task usually involves many steps. An agent needs to know what they are and plan ahead.\nTask Decomposition#\nChain of thought (CoT; Wei et al. 2022) has become a standard prompting technique for enhancing model performance on complex tasks. The model is instructed to “think step by step” to utilize more test-time computation to decompose hard tasks into smaller and simpler steps. CoT transforms big tasks into multiple manageable tasks and shed lights into an interpretation of the model’s thinking process.\nTree of Thoughts (Yao et al. 2023) extends CoT by exploring multiple reasoning possibilities at each step. It first decomposes the problem into multiple thought steps and generates multiple thoughts per step, creating a tree structure. The search process can be BFS (breadth-first search

In [31]:
len(docs)

1

# Part 4 : Generation

In [38]:
# PROMPT

# Required packages :
# import google.generativeai as genai
# from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

template = """Answer the question based only on the following context:
{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)
prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Answer the question based only on the following context:\n{context}\n\nQuestion: {question}\n'), additional_kwargs={})])

In [39]:
# LLM
llm = llm

In [40]:
# Chain
chain = prompt | llm

In [41]:
# Run
chain.invoke({"context":docs,"question":"What is Task Decomposition?"})

AIMessage(content='Task decomposition is the process of breaking down a complicated task into smaller and simpler steps. It can be done by LLM with simple prompting, by using task-specific instructions, or with human inputs.', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': []}, id='run-4c4e6cfa-99e7-4bbd-b034-654e33e7937d-0', usage_metadata={'input_tokens': 341, 'output_tokens': 41, 'total_tokens': 382, 'input_token_details': {'cache_read': 0}})

In [45]:
# Sample rag-prompt with placeholders for context and question
from langchain import hub
prompt_hub_rag = hub.pull("rlm/rag-prompt")
prompt_hub_rag

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, metadata={'lc_hub_owner': 'rlm', 'lc_hub_repo': 'rag-prompt', 'lc_hub_commit_hash': '50442af133e61576e74536c6556cefe1fac147cad032f4377b60c436e6cdcb6e'}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\nQuestion: {question} \nContext: {context} \nAnswer:"), additional_kwargs={})])

In [47]:
# CHAIN RUNNABLES

from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

rag_chain.invoke("What is Task Decomposition?")

'Task decomposition is the process of breaking down a complicated task into smaller and simpler steps. It can be done by LLM with simple prompting, by using task-specific instructions, or with human inputs.'